here we will create our first rag pipelin.
in the previous we stored the vector and retrieved similar to the query. This is only a list of payloads but not an answer 

In [2]:
import openai
from qdrant_client import QdrantClient

we will be build a vector embedding for query 
retrieve relevant Data
run a generation run against an llm to build a proper answer for the user

### Embedding Function

In [1]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval Function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [11]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-01",
        query=query_embedding, 
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scored = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocess_description"])
        similarity_scored.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])


    return {
        "retrieved_context_ids":  retrieved_context_ids, 
        "retrieved_context": retrieved_context,
        "similarity_scored": similarity_scored,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [12]:
retrieved_context = retrieve_data("do you have any usb fans for the summer heat?", k=10)

In [13]:
retrieved_context

{'retrieved_context_ids': ['B0BRJS644Z',
  'B0BXC72RLD',
  'B099N9F3FP',
  'B0CF57H28T',
  'B0C8DBH7ZT',
  'B0BM9THPDQ',
  'B0C9QZS95R',
  'B09VDLH5M6',
  'B0BYD7PGV1',
  'B0C996WY16'],
 'retrieved_context': ['Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm)【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that suppo

we can inject the above info into the generation step or we can explose some of this info for observabilty and debugging. 
i.e we can look at the similarity scores:

0.55251706,
0.5322559,
0.45835388,
0.34993696,
0.34819797,

in this case the similarity score drops off after the first three matches. So may not be as relevant to the user. 

### Format retrieved context function

In [18]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [19]:
preprocessed_context = process_context(retrieved_context)

In [21]:
print(preprocessed_context)

- ID: B0BRJS644Z, rating: 4.7, description: Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm)【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】 Rubber Feet on the fan raise it up enough off of the surface it is sittin

now we have the preprocessed context above, a list of the retrieved items with id, rating, description
we are formatting the injected context to be relevant
so the model has more context to answer the users query

we can use the id to ground the answer in specific items -> i.e so it can surface the correct item to the user and display when relevant 

### create prompt template function

In [29]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
    """

    return prompt

In [30]:
prompt = build_prompt(preprocessed_context, "Do you have a usb fan that will help me cool down in hot summers?")

In [31]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- ID: B0BRJS644Z, rating: 4.7, description: Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm)【Solve Your Cooling Problem】 This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed Control Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility US


### Generate answer function

In [33]:
from urllib3 import response


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ], 
        reasoning_effort="none"
    )

    return response.choices[0].message.content

In [34]:
print(generate_answer(prompt))

Yes. We have USB fans that are designed to cool down during hot weather:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (120mm x 120mm x 55mm)
- Speed control on the cord: off/low/medium/high
- USB-powered via a 3.3 ft USB cable (works with PCs/laptops/power banks/AC adapters/car chargers with USB)
- Built for keeping devices like router/modem/DVR/Xbox TV boxes cool

2) HZD Desk Fan Rechargeable, Mini Portable Fan (3 speeds)
- 3 speeds: low/medium/high
- USB-powered (it comes with a USB cable; note: this model does not include a battery)
- Quiet operation (noise less than 50d)
- Compact for home, office, travel, camping, etc.

If you tell me where you’ll use it (desk/table vs. cooling an electronics device like a TV box/router), I can recommend the best match.


### Combined RAG pipeline

In [ ]:
def rag_pipeline(question, topk_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, k=topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer


In [37]:
print(rag_pipeline("Do you have a usb fan that will help me cool down in hot summers?"))

Yes. We have two USB fan options that can help cool you down in hot summers:

1) Marame 120mm 5V USB Powered Fan with Speed Controller (ID: B0BRJS644Z, rating 4.7)
- Adjustable speed via an on-cord controller: off / low / medium / high
- Powered by a USB cable (3.3 ft)
- Works well for cooling devices in confined spaces (like router/modem/Xbox cabinets)

2) HZD Desk Fan Rechargeable, Mini Portable Fan (ID: B0BXC72RLD, rating 4.3)
- Compact personal desk fan with 3 speeds (low/medium/high)
- Noise is listed as less than 50 dB
- USB powered (USB cable included, 4.9 ft); note: this one does not come with a battery
- Suitable for home, office, travel, and camping

Which do you want: a larger 120mm fan for cooling electronics, or a small portable desk fan for personal cooling?
